# Context

This notebook was developed as a report of my work in the discipline of Video Coding, given by the professors, Ismael Seidel and José Luiz Almada Güntzel. 
It is about Light Fields, where I apply some estimatives of the objective metric Structural Similarity Index Measure (SSIM) comparing always with the real calculated values. 

In [ ]:
from IPython.display import display
import json
import io
import numpy as np
import os
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from PIL import Image
import plotly.express as px
from skimage.metrics import mean_squared_error as mse
from skimage.util import crop
from scipy import ndimage, datasets
from scipy.ndimage import uniform_filter, gaussian_filter
from tqdm import tqdm
import sys
import yuvio
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({"axes.spines.top": False,
                     "axes.spines.right": False})

In [ ]:
try:
    open('drey.txt')
    path_raw = '../lfctest/ce/RAW/lightfields/YUV/serpentine/'
except:
    path_raw = '/home/andrey/lfctest/ce/RAW/lightfields/YUV/serpentine/'

# Intro

Let's start by exploring the directories and looking for the reports from the decoded Light Fields (LF). In the next cells, I want to show the path to get each one.

In [ ]:
# Here we can see some of the LFs decoded with lfctoolkit. These jsons contain different metrics for each LF, like SSIM, PSNR, ... 
# I will pass their names to access later with python
!ls ../jpeg-pleno-ctc-tools/lfc_toolkit/examples/results/jplm/metrics_report/vmaf/ | head
!ls ../jpeg-pleno-ctc-tools/lfc_toolkit/examples/results/jplm/metrics_report/vmaf/ > txt.txt

In [ ]:
# this cell opens txt.txt, reads each line and pass each one for a list of names
path_jpeg = '../jpeg-pleno-ctc-tools/lfc_toolkit/examples/results/jplm/metrics_report/vmaf/'

f = open('txt.txt', 'r')
texto = f.read()
all_LFs = texto.split('\n')
all_LFs = [i for i in all_LFs if i.endswith('.json')]
all_LFs[:10]

\
Here I store in the LightField_names variable just the name of each LF + its compress tax.

In [ ]:
LightField_names = [lf.split('_vmaf')[0] for lf in all_LFs]
LightField_names[:10]

In [ ]:
# Nessa célula eu obtenho todos os valores do PSNR_y e SSIM
# para cada view de cada LF, que foi rodada pelo lfctoolkit.
# PS: não sei como o SSIM foi calculado
ssim_dict_of_each_LF = {}
psnr_dict_of_each_LF = {}

for each_LF, LF_name in zip(all_LFs, LightField_names):
    ssim_list_of_each_view_per_LF = []
    psnr_list_of_each_view_per_LF = []
    
    # Lembrando path_jpeg = '../jpeg-pleno-ctc-tools/lfc_toolkit/examples/results/jplm/metrics_report/vmaf/'
    path_corrigido = os.path.join(path_jpeg + each_LF)
    
    with open(path_corrigido) as f:
        
        for linha in f:  # 1 line is exactly like '         "integer_adm_scale0": 0.888848,'
            par = linha.strip()[:-1].split(': ')  # which par is ['"integer_adm_scale0"', '0.888848']

            try:
                if par[0] == '"float_ssim"':
                    ssim_list_of_each_view_per_LF.append(float(par[1]))
                elif par[0] == '"psnr_y"':
                    psnr_list_of_each_view_per_LF.append(float(par[1]))
                    
            except:
                continue
                
    ssim_dict_of_each_LF[LF_name] = ssim_list_of_each_view_per_LF
    psnr_dict_of_each_LF[LF_name] = psnr_list_of_each_view_per_LF

mean_psnr_y_bikes015 = np.mean(psnr_dict_of_each_LF['Bikes_015'])
print('mean_psnr_y_bikes015 ', mean_psnr_y_bikes015) # verified some cells bellow

mean_ssim_bikes015 = np.mean(ssim_dict_of_each_LF['Bikes_015'])
print('mean_ssim_bikes015 ', mean_ssim_bikes015) # verified some cells below
print('\nBikes_015:', psnr_dict_of_each_LF['Bikes_015'])

\
Quero fazer um DataFrame com todos os dados, ou seja, cujas linhas sejam cada Light Field com sua respectiva taxa, e cada coluna um valor do SSIM ou do PSNR_y de cada view. Como os LFs tem quantidades de views diferentes, vou completá-los com nulos pra terem as mesmas dimensões

In [ ]:
ssim_df = pd.DataFrame.from_dict(ssim_dict_of_each_LF, orient = 'index')
ssim_df.columns = [f'view_{i}' for i in ssim_df.columns]

ssim_df['media'] = ssim_df.mean(axis = 1, skipna = True)
ssim_df['std'] = ssim_df.drop(columns=['media']).std(axis=1, skipna=True)
print(ssim_df.shape)
ssim_df.index.name = 'ssim'
ssim_df[['view_0', 'view_1', 'view_2', 'view_3', 'media']].head(10)

Esse dataframe é o ssim de cada view de cada LF

\
Agora vou fazer o Data Frame pro PSNR e pro MSE, calculado diretamente do PSNR

In [ ]:
psnr_df = pd.DataFrame.from_dict(psnr_dict_of_each_LF, orient = 'index')
psnr_df.columns = [f'view_{i}' for i in psnr_df.columns]
psnr_df.index.name = 'psnr'
psnr_df['media'] = psnr_df.mean(axis = 1, skipna = True)
psnr_df[['view_0', 'view_1', 'view_2', 'view_3', 'media']].head(5)

In [ ]:
def MSE(original, pred):
    mse = np.mean(np.square(original.astype(np.float64) - pred.astype(np.float64)))
    return mse
    
def mse_from_psnr(psnr, b):
    mse = ((2**b -1)**2)/10**(psnr/10)
    return mse

In [ ]:
mse_df = psnr_df.apply(mse_from_psnr, b = 10)
mse_df = mse_df.drop(columns = ['media'])
mse_df.index.name = 'mse'
mse_df['media'] = mse_df.mean(axis = 1, skipna = True)
mse_df[['view_0', 'view_1', 'view_2', 'view_3', 'media']].head()

Pra verificar se fiz corretamente, podemos ver no json do `Bikes_015` que tanto a média do PSNR_y quanto o SSIM batem com o valor dos dfs

In [ ]:
# !cd ../jpeg-pleno-ctc-tools/lfc_toolkit/examples/results/jplm/metrics_report/vmaf/\
# && tail -n 34 Bikes_015_vmaf_11_11_625x434_yuv444p10le.json\
# | head -n 5\
# && tail -n 16 Bikes_015_vmaf_11_11_625x434_yuv444p10le.json\
# | head -n 5

---


In [ ]:
# !cd ../lfctest/ce/results/jplm_ssim_estimate_results/ && echo */logs/*.json

In [ ]:
_ = 'Bikes/logs/Bikes_015_11_11_625x434_PGX.json Bikes/logs/Bikes_03_11_11_625x434_PGX.json Bikes/logs/Bikes_12_11_11_625x434_PGX.json Bikes/logs/Bikes_3_11_11_625x434_PGX.json Bikes/logs/Bikes_96_11_11_625x434_PGX.json Danger_de_Mort/logs/Danger_de_Mort_015_11_11_625x434_PGX.json Danger_de_Mort/logs/Danger_de_Mort_03_11_11_625x434_PGX.json Danger_de_Mort/logs/Danger_de_Mort_12_11_11_625x434_PGX.json Danger_de_Mort/logs/Danger_de_Mort_3_11_11_625x434_PGX.json Danger_de_Mort/logs/Danger_de_Mort_96_11_11_625x434_PGX.json Fountain_Vincent2/logs/Fountain_Vincent2_015_11_11_625x434_PGX.json Fountain_Vincent2/logs/Fountain_Vincent2_03_11_11_625x434_PGX.json Fountain_Vincent2/logs/Fountain_Vincent2_12_11_11_625x434_PGX.json Fountain_Vincent2/logs/Fountain_Vincent2_3_11_11_625x434_PGX.json Fountain_Vincent2/logs/Fountain_Vincent2_96_11_11_625x434_PGX.json greek/logs/greek_005_9_9_512x512_PGX.json greek/logs/greek_0075_9_9_512x512_PGX.json greek/logs/greek_015_9_9_512x512_PGX.json greek/logs/greek_03_9_9_512x512_PGX.json greek/logs/greek_12_9_9_512x512_PGX.json set2/logs/set2_005_33_11_1920x1080_PGX.json set2/logs/set2_0075_33_11_1920x1080_PGX.json set2/logs/set2_015_33_11_1920x1080_PGX.json set2/logs/set2_03_33_11_1920x1080_PGX.json set2/logs/set2_12_33_11_1920x1080_PGX.json sideboard/logs/sideboard_015_9_9_512x512_PGX.json sideboard/logs/sideboard_03_9_9_512x512_PGX.json sideboard/logs/sideboard_12_9_9_512x512_PGX.json sideboard/logs/sideboard_3_9_9_512x512_PGX.json sideboard/logs/sideboard_96_9_9_512x512_PGX.json Stone_Pillars_Outside/logs/Stone_Pillars_Outside_015_11_11_625x434_PGX.json Stone_Pillars_Outside/logs/Stone_Pillars_Outside_03_11_11_625x434_PGX.json Stone_Pillars_Outside/logs/Stone_Pillars_Outside_12_11_11_625x434_PGX.json Stone_Pillars_Outside/logs/Stone_Pillars_Outside_3_11_11_625x434_PGX.json Stone_Pillars_Outside/logs/Stone_Pillars_Outside_96_11_11_625x434_PGX.json tarot/logs/tarot_005_17_17_1024x1024_PGX.json tarot/logs/tarot_0075_17_17_1024x1024_PGX.json tarot/logs/tarot_015_17_17_1024x1024_PGX.json tarot/logs/tarot_03_17_17_1024x1024_PGX.json tarot/logs/tarot_12_17_17_1024x1024_PGX.json'
LFs = _.split(' ')

path_lfc = '../lfctest/ce/results/jplm_ssim_estimate_results/'

LFs[:5]

In [ ]:
# Nessa célula eu coleto o SSIM estimado para os LFs, no geral, e não por view.
ssim_estimados_dict_por_lf = {}

for lf in LFs:

    path_corrigido = os.path.join(path_lfc + lf)     
    lf_nome = '_'.join(lf.split('/')[2].split('_')[:-4]) # Lembrando que cada lf é um elemento
    # da celula anterior. Ex: 'Bikes/logs/Bikes_03_11_11_625x434_PGX.json'
    # print(lf_nome_lista) retorna ['Bikes', '015']
    

        
    with open(path_corrigido, 'r') as f:
        
        for line in f:
            par = line.strip().split(': ')  # ['"est_ssim"', '0.9830602176303537']
            
            try:
                ssim_estimados_dict_por_lf[lf_nome] = float(par[1])
            
            except:
                continue

ssim_estimados_df = pd.DataFrame(ssim_estimados_dict_por_lf, index = ['ssim_estimados_dict_por_lf']).T
ssim_estimados_df.columns = ['ssim_estimado']
ssim_estimados_df.head()

In [ ]:
df_merged = pd.concat([ssim_estimados_df, ssim_df[['media']]], axis=1, join='outer')
df_merged.columns = ['ssim_estimado', 'ssim_original']
df_merged.head()

---

# Estimações de SSIM para Light Field

A partir daqui, vou implementar estimações de SSIM que desenvolvi para reproduzir os dados do luc e da martini no outro notebook. Algumas alterações serão feitas para incluir o formato dos LFs.

Como eu desenvolvi muitas das funções para compressão de imagem usando MSE em outro notebook, e aqui eu já tenho o $PSNR_y$ de cada view, apenas vou transformar o PSNR em MSE usando a relação de MSE em função de PSNR:


$$
MSE = \frac{(2^b-1)^2}{10^{\text{PSNR}/10}}
$$

A primeira equação de estimação do SSIM, deduzida pela Maria Martini é 

$$
SSIM \approx 1-\frac{MSE_l}{(2\sigma_x^2 + c_2)}
$$

onde tanto $MSE_l$ quanto $\sigma_x^2$ são locais. O próximo passado dos autores no paper é estimar o valor de $MSE_l$ com outras estatísticas. Pra isso, primeiro eles atribuem um parâmetro livre $\beta$. Que aqui será calculado como $0.5$, em decorrência da aproximação feita no paper, e depois tentatei estimar um valor mais adequado para LFs.

A primeira equação de estimação pro $MSE_l$ com o $\beta$ fica:

$$MSE_l \approx MSE_G \frac{(\sigma_x^2 + \epsilon)^\beta}{\mathbb{E}[(\sigma_x^2+\epsilon)^\beta]}$$

onde $\epsilon\ =\ 1e^{-6}$

Para a aproximação feita no paper, $\beta \approx 0.5$, a variância se tornar o desvio padrão:

$$MSE_l \approx MSE_G \frac{(\sigma_x + \epsilon)}{\mathbb{E}[\sigma_x + \epsilon]}$$

Trocando na equação inicial, fica:

$$
SSIM \approx 1-\frac{MSE_G}{(2\sigma_x^2 + c_2)}\frac{(\sigma_x + \epsilon)}{\mathbb{E}[\sigma_x + \epsilon]}
$$

E o SSIM global é calculado pela média de todas as janelas do frame da imagem original.

$$
MSSIM(X, Y) \approx \frac{1}{M}\sum_j \left ( 1-\frac{MSE_G}{(2\sigma_x^2 + c_2)}\frac{(\sigma_x + \epsilon)}{\mathbb{E}[\sigma_x + \epsilon]} \right)
$$

\
Vou começar com o Bikes. Na célula a seguir, vamos os light fields decodados. 

In [ ]:
# !ls  ../lfctest/ce/results/jplm_ssim_estimate_results/Bikes/decoded/yuv/

\
A seguir está a nova função de estimar a ssim a partir da $MSE_G$ para light fields. Detalhe, global aqui quer dizer por view. Como visto antes, a MSE foi obtida do PSNR, que foi calculado usando a view inteira. Já a variância da imagem original também não foi usada. Como a martini diz no seu paper, a variância é uma das medidas que muda pouco para compressão via DCT. Nesse sentido, usei a variância local de cada view decodada.

Logo, será a implementação da seguinte função:

$$
SSIM \approx 1-\frac{MSE_G}{(2\sigma_x^2 + c_2)}
$$

In [ ]:
from funcoes import calcular_ssim_df

## Implementando a SSIM do Luc

In [ ]:
from funcoes import ssim_luc

In [ ]:
d = {
    'Bikes': ['Bikes_015_11_11_625x434_yuv444p10le.yuv', 'Bikes_03_11_11_625x434_yuv444p10le.yuv', 'Bikes_12_11_11_625x434_yuv444p10le.yuv', 'Bikes_3_11_11_625x434_yuv444p10le.yuv', 'Bikes_96_11_11_625x434_yuv444p10le.yuv'],
    'Danger_de_Mort': ['Danger_de_Mort_015_11_11_625x434_yuv444p10le.yuv', 'Danger_de_Mort_03_11_11_625x434_yuv444p10le.yuv', 'Danger_de_Mort_12_11_11_625x434_yuv444p10le.yuv',  'Danger_de_Mort_3_11_11_625x434_yuv444p10le.yuv', 'Danger_de_Mort_96_11_11_625x434_yuv444p10le.yuv'],
    'Fountain_Vincent2': ['Fountain_Vincent2_015_11_11_625x434_yuv444p10le.yuv', 'Fountain_Vincent2_03_11_11_625x434_yuv444p10le.yuv', 'Fountain_Vincent2_12_11_11_625x434_yuv444p10le.yuv', 'Fountain_Vincent2_3_11_11_625x434_yuv444p10le.yuv', 'Fountain_Vincent2_96_11_11_625x434_yuv444p10le.yuv'],
    'greek': ['greek_005_9_9_512x512_yuv444p10le.yuv', 'greek_0075_9_9_512x512_yuv444p10le.yuv', 'greek_015_9_9_512x512_yuv444p10le.yuv', 'greek_03_9_9_512x512_yuv444p10le.yuv', 'greek_12_9_9_512x512_yuv444p10le.yuv'],
    # 'set2': ['set2_03_33_11_1920x1080_yuv444p10le.yuv', 'set2_005_33_11_1920x1080_yuv444p10le.yuv', 'set2_12_33_11_1920x1080_yuv444p10le.yuv', 'set2_015_33_11_1920x1080_yuv444p10le.yuv',  'set2_0075_33_11_1920x1080_yuv444p10le.yuv'],
}

In [ ]:
dfs = []

for lf_nome, LFs in d.items():
    path_dir_yuv = f'../lfctest/ce/results/jplm_ssim_estimate_results/{lf_nome}/decoded/yuv/'
    df = calcular_ssim_df(ssim_luc, LFs = LFs, nome = 'ssim_luc')
    dfs.append(df)

ssim_luc = pd.concat(dfs)
ssim_luc['media'] = ssim_luc.mean(axis = 1, skipna = True)
ssim_luc['std'] = ssim_luc.drop(columns=['media']).std(axis=1, skipna=True)

In [ ]:
ssim_luc.head(10)

In [ ]:
dfs = []

for lf_nome, LFs in d.items():
    path_dir_yuv = f'../lfctest/ce/results/jplm_ssim_estimate_results/{lf_nome}/decoded/yuv/'
    df = calcular_ssim_df
(ssim_luc, LFs = LFs, filtro = 'gaussian')
    dfs.append(df)

ssim_gaussian = pd.concat(dfs)
ssim_gaussian['media'] = ssim_gaussian.mean(axis = 1, skipna = True)
ssim_gaussian['std'] = ssim_gaussian.drop(columns=['media']).std(axis=1, skipna=True)

## Implementando a estimação da Martini

A equação a ser implementada é

$$SSIM \approx 1-\frac{MSE_l}{(2\sigma_x^2 + c_2)}$$

In [ ]:
from funcoes import ssim_martini

In [ ]:
dfs = []

for lf_nome, LFs in d.items():
    path_dir_yuv = f'../lfctest/ce/results/jplm_ssim_estimate_results/{lf_nome}/decoded/yuv/'
    df = calcular_ssim_df
(ssim_martini_lf, LFs = LFs)
    dfs.append(df)

ssim_martini = pd.concat(dfs)
ssim_martini['media'] = ssim_martini.mean(axis = 1, skipna = True)
ssim_martini['std'] = ssim_martini.drop(columns=['media']).std(axis=1, skipna=True)

## Implementando com $MSE_G$

Lembrando que a equação a ser implementada é

$$
SSIM \approx 1-\frac{MSE_G}{(2\sigma_x^2 + c_2)}
$$

In [ ]:
from funcoes import ssim_global

In [ ]:
dfs = []

for lf_nome, LFs in d.items():
    path_dir_yuv = f'../lfctest/ce/results/jplm_ssim_estimate_results/{lf_nome}/decoded/yuv/'
    df = calcular_ssim_df
(ssim_global_lf, LFs = LFs)
    dfs.append(df)

ssim_global = pd.concat(dfs)
ssim_global['media'] = ssim_global.mean(axis = 1, skipna = True)
ssim_global['std'] = ssim_global.drop(columns=['media']).std(axis=1, skipna=True)

## Implementação com a $MSE_l$ estimada com $\beta = 0.5$

A equação a ser implementada é

$$MSE_l \approx MSE_G \frac{(\sigma_x^2 + \epsilon)^\beta}{\mathbb{E}[(\sigma_x^2+\epsilon)^\beta]}$$

In [ ]:
from funcoes import ssim_beta

In [ ]:
dfs = []

for lf_nome, LFs in d.items():
    path_dir_yuv = f'../lfctest/ce/results/jplm_ssim_estimate_results/{lf_nome}/decoded/yuv/'
    df = calcular_ssim_df
(ssim_beta_lf, LFs = LFs)
    dfs.append(df)

ssim_beta050 = pd.concat(dfs)
ssim_beta050.index.name = 'beta050'
ssim_beta050['media'] = ssim_beta050.mean(axis = 1, skipna = True)
ssim_beta050['std'] = ssim_beta050.drop(columns=['media']).std(axis=1, skipna=True)

In [ ]:
# Corrigindo o index da ssim_df
ssim_df.columns = list(range(ssim_df.shape[1] - 2)) + ['media', 'std']

## Resultados: Comparando Implementações e erros relativos

In [ ]:
from funcoes import comparando_implementacoes_lf, comparando_erros_lf

In [ ]:
dfs = [ssim_df, ssim_luc, ssim_gaussian, ssim_martini, ssim_global, ssim_beta050]
colors = ['gray', '#000', '#800080', '#00f', '#ff8c00', '#ff0000']
labels = ['ssim vmaf', 'ssim luc', 'ssim gaussiana', 'ssim martini', 'ssim global', r'ssim $\beta = 0.5$']

dfs_erro = [ssim_luc, ssim_gaussian, ssim_martini, ssim_global, ssim_beta050]
colors_erro = ['#000', '#800080', '#00f', '#ff8c00', '#ff0000']
labels_erro = ['ssim luc', 'ssim gaussiana', 'ssim martini', 'ssim global', r'ssim $\beta = 0.5$'] 
txs = [1.5, 3, 12, 30, 96]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

comparar_implementacoes(dfs, colors, labels, 'Bikes', 0, 5, txs=txs, ax=axes[0])
comparando_erros(dfs_erro, labels_erro, colors_erro, 0, 5, txs=txs, ax=axes[1])

plt.tight_layout()
plt.savefig('Bikes.jpeg', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

comparar_implementacoes(dfs, colors, labels, 'Danger de Mort', 5, 10, txs=txs, ax=axes[0])
comparando_erros(dfs_erro, labels_erro, colors_erro, 5, 10, txs=txs, ax=axes[1])

plt.tight_layout()
plt.savefig('Danger de Mort.jpeg', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

comparar_implementacoes(dfs, colors, labels, 'Fountain Vincent12', 10, 15, txs=txs, ax=axes[0])
comparando_erros(dfs_erro, labels_erro, colors_erro, 10, 15, txs=txs, ax=axes[1])

plt.tight_layout()
plt.savefig('Fountain Vincent12.jpeg', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
txs_g = [0.5, 0.75, 1.5, 3, 12]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

comparar_implementacoes(dfs, colors, labels, 'greek', 15, 20, txs=txs_g, ax=axes[0])
comparando_erros(dfs_erro, labels_erro, colors_erro, 10, 15, txs=txs_g, ax=axes[1])

plt.tight_layout()
plt.savefig('greek.jpeg', dpi = 300, bbox_inches = 'tight')
plt.show()

## Encontrando um beta mais adequado

## Teste para $\beta = 1$

In [ ]:
dfs = []

for lf_nome, LFs in d.items():
    path_dir_yuv = f'../lfctest/ce/results/jplm_ssim_estimate_results/{lf_nome}/decoded/yuv/'
    df = calcular_ssim_df
(ssim_beta_lf, LFs = LFs, beta = 1)
    dfs.append(df)

ssim_beta1 = pd.concat(dfs)
ssim_beta1.index.name = 'beta1'
ssim_beta1['media'] = ssim_beta1.mean(axis = 1, skipna = True)
ssim_beta1['std'] = ssim_beta1.drop(columns=['media']).std(axis=1, skipna=True)

In [ ]:
dfs = [ssim_df, ssim_beta1]
colors = ['gray', '#ff0000']
labels = ['ssim vmaf', r'ssim $\beta = 1$']

dfs_erro = [ssim_beta1]
colors_erro = ['#ff0000']
labels_erro = [r'ssim $\beta = 1$'] 
txs = [1.5, 3, 12, 30, 96]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

comparar_implementacoes(dfs, colors, labels, 'Bikes', 0, 5, txs=txs, ax=axes[0])
comparando_erros(dfs_erro, labels_erro, colors_erro, 0, 5, txs=txs, ax=axes[1])

plt.tight_layout()
plt.savefig('Bikes_beta1.jpeg', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

comparar_implementacoes(dfs, colors, labels, 'Danger de Mort', 5, 10, txs=txs, ax=axes[0])
comparando_erros(dfs_erro, labels_erro, colors_erro, 5, 10, txs=txs, ax=axes[1])

plt.tight_layout()
plt.savefig('Danger de Mort_beta1.jpeg', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

comparar_implementacoes(dfs, colors, labels, 'Fountain Vincent12', 10, 15, txs=txs, ax=axes[0])
comparando_erros(dfs_erro, labels_erro, colors_erro, 10, 15, txs=txs, ax=axes[1])

plt.tight_layout()
plt.savefig('Fontain Vincent12_beta1.jpeg', dpi = 300, bbox_inches = 'tight')
plt.show()

In [ ]:
txs_g = [0.5, 0.75, 1.5, 3, 12]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

comparar_implementacoes(dfs, colors, labels, 'greek', 15, 20, txs=txs_g, ax=axes[0])
comparando_erros(dfs_erro, labels_erro, colors_erro, 10, 15, txs=txs_g, ax=axes[1])

plt.tight_layout()
plt.savefig('greek_beta1.jpeg', dpi = 300, bbox_inches = 'tight')
plt.show()